# 2장 02. 반복 학습, 평가, 기록 추적

이 notebook은 모델 후보를 여러 번 학습하고, 각 실행의 평가 결과를 run 기록으로 남기는 과정을 봅니다.

앞의 `01_score_threshold.ipynb`가 score와 threshold의 뜻을 보는 실습이었다면, 이 notebook은 그 score를 만드는 모델을 바꿔 가며 비교합니다. 목표는 가장 좋은 모델을 찾는 것이 아니라, **학습 실행마다 같은 기준으로 평가하고 기록을 남기는 흐름**을 이해하는 것입니다.


## 기본 개념: train run과 tracking

모델 학습은 한 번 실행하고 끝나는 일이 아닙니다. feature가 같아도 모델 종류, hyperparameter, 학습 데이터 버전이 바뀌면 결과가 달라집니다. 그래서 MLOps에서는 실행 하나를 `run`으로 보고, run마다 조건과 결과를 함께 남깁니다.

| 항목 | 쉬운 뜻 | 이번 notebook에서 보는 값 |
| --- | --- | --- |
| train data | 모델이 기준을 배우는 데이터 | 학습 row 수, label 분포 |
| candidate | 이번에 시도한 모델 후보 | `rf_small`, `rf_deeper`, `logistic_baseline` |
| params | 학습 조건 | 모델 종류, tree 깊이, threshold |
| metrics | 평가 결과 | precision, recall, F1, FP, FN |
| artifact | 나중에 다시 찾을 결과물 | model artifact 이름, JSON run 기록 |

MLflow를 쓰면 이 run 기록이 MLflow 서버에 저장됩니다. 여기서는 먼저 notebook 안에서 작은 tracking table을 만들고, 같은 생각이 MLflow 기록으로 어떻게 이어지는지 확인합니다.


In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

import utils as runtime

prepared = await runtime.prepare_notebook()
pd = prepared.pandas
aiq = prepared.aiq_lite
await runtime.ensure_sklearn()

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline


## 1. 학습/평가 데이터 준비

먼저 데이터를 읽습니다. 로컬에서 `data/vital_signs_train.csv`, `data/vital_signs_test.csv`가 있으면 그 파일을 사용하고, 브라우저 실습에서는 준비된 작은 sample data를 사용합니다.

이 단계에서 확인할 핵심은 모델 학습과 평가를 같은 데이터로 하지 않는다는 점입니다. 학습 데이터는 모델이 기준을 배우는 데 쓰고, 평가 데이터는 학습 후 성능을 확인하는 데 씁니다.


In [ ]:
raw_data, source = aiq.load_csv_or_sample(
    "data/vital_signs_train.csv",
    aiq.sample_vital_signs(),
    nrows=200,
)
raw_data = raw_data.reset_index(drop=True)

split_index = max(4, int(len(raw_data) * 0.7))
train_df = raw_data.iloc[:split_index].copy()
test_df = raw_data.iloc[split_index:].copy()

print("source:", source)
print("train_rows:", len(train_df))
print("test_rows:", len(test_df))


**출력 확인**

`source`가 `data/vital_signs_train.csv`이면 로컬 준비 데이터를 읽은 것입니다. `sample` 또는 fallback으로 표시되면 브라우저용 작은 데이터를 사용한 것입니다.

중요한 것은 row 수 자체가 아니라 train/test가 분리되어 있다는 점입니다. 같은 데이터로 학습하고 같은 데이터로 평가하면 모델이 외운 결과를 성능처럼 착각할 수 있습니다.


In [ ]:
label_summary = pd.DataFrame(
    [
        {"split": "train", "label": label, "count": count}
        for label, count in train_df["label"].value_counts().items()
    ]
    + [
        {"split": "test", "label": label, "count": count}
        for label, count in test_df["label"].value_counts().items()
    ]
)
label_summary


**출력 확인**

이 표는 train/test에 `high_risk`, `low_risk` label이 얼마나 있는지 보여 줍니다. 평가 데이터에 positive label인 `high_risk`가 거의 없으면 recall 해석이 불안정해집니다.

초급 실습에서는 label 분포가 완벽할 필요는 없습니다. 다만 metric을 읽기 전에 “평가할 정답이 어떤 분포로 들어 있는가”를 먼저 봐야 합니다.


## 2. 모델 후보 정하기

이제 세 가지 후보를 만듭니다. 후보가 달라지면 score와 prediction도 달라질 수 있으므로, 모델 이름과 parameter를 run 기록에 남겨야 합니다.


In [ ]:
candidates = pd.DataFrame(
    [
        {"run_id": "run_001", "model_name": "rf_small", "model_type": "random_forest", "n_estimators": 30, "max_depth": 2, "threshold": 0.50},
        {"run_id": "run_002", "model_name": "rf_deeper", "model_type": "random_forest", "n_estimators": 80, "max_depth": 4, "threshold": 0.50},
        {"run_id": "run_003", "model_name": "logistic_baseline", "model_type": "logistic_regression", "n_estimators": None, "max_depth": None, "threshold": 0.50},
    ]
)
candidates


**출력 확인**

각 행이 하나의 학습 실행 후보입니다. `run_id`는 나중에 metric, model artifact, 배포 후보를 연결할 때 쓰는 이름입니다.

여기서 중요한 점은 모델 후보를 말로만 비교하지 않고 표로 남긴다는 점입니다. 이 표가 MLflow의 params 화면에 해당한다고 보면 됩니다.


## 3. 첫 번째 후보를 실제로 학습하고 평가하기

먼저 후보 하나만 실행합니다. 여기서 `fit()`이 실제 학습이고, `predict_proba()`가 score 생성입니다. 그 다음 threshold를 적용해 prediction을 만들고, label과 비교해 metric을 계산합니다.


In [ ]:
FEATURE_COLUMNS = list(aiq.FEATURE_COLUMNS)
POSITIVE_LABEL = aiq.POSITIVE_LABEL
THRESHOLD = 0.50


def make_model(config: dict):
    if config["model_type"] == "random_forest":
        estimator = RandomForestClassifier(
            n_estimators=int(config["n_estimators"]),
            max_depth=int(config["max_depth"]),
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=42,
            n_jobs=1,
        )
    else:
        estimator = LogisticRegression(max_iter=500, class_weight="balanced")

    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("model", estimator),
        ]
    )

first_config = candidates.iloc[0].to_dict()
first_model = make_model(first_config)
first_model.fit(train_df[FEATURE_COLUMNS], train_df["label"].eq(POSITIVE_LABEL).astype(int))

first_scores = first_model.predict_proba(test_df[FEATURE_COLUMNS])[:, 1]
first_predictions = [POSITIVE_LABEL if score >= THRESHOLD else aiq.NEGATIVE_LABEL for score in first_scores]

preview = test_df[["label", *FEATURE_COLUMNS]].head(5).copy()
preview["score"] = [round(float(score), 4) for score in first_scores[:5]]
preview["prediction"] = first_predictions[:5]
preview


**출력 확인**

이 표는 모델이 평가 데이터에 대해 만든 score와 prediction을 보여 줍니다. `label`은 정답이고, `prediction`은 모델이 threshold 기준으로 낸 결과입니다.

score가 0.50 이상이면 `high_risk`, 0.50보다 낮으면 `low_risk`로 바뀝니다. 이 변환 기준도 run 기록에 남겨야 나중에 같은 모델을 다시 해석할 수 있습니다.


In [ ]:
true_binary = test_df["label"].eq(POSITIVE_LABEL).astype(int)
pred_binary = pd.Series(first_predictions).eq(POSITIVE_LABEL).astype(int)
tn, fp, fn, tp = confusion_matrix(true_binary, pred_binary, labels=[0, 1]).ravel()

first_metrics = pd.DataFrame(
    [
        {"metric": "accuracy", "value": accuracy_score(true_binary, pred_binary)},
        {"metric": "precision", "value": precision_score(true_binary, pred_binary, zero_division=0)},
        {"metric": "recall", "value": recall_score(true_binary, pred_binary, zero_division=0)},
        {"metric": "f1", "value": f1_score(true_binary, pred_binary, zero_division=0)},
        {"metric": "false_positive", "value": fp},
        {"metric": "false_negative", "value": fn},
    ]
)
first_metrics


**출력 확인**

이 표가 한 run의 평가 결과입니다. Precision과 recall은 단독으로 좋고 나쁨을 말하기보다 FP/FN과 함께 봐야 합니다.

예를 들어 recall이 낮으면 실제 `high_risk`를 놓친 FN이 많을 수 있습니다. 반대로 precision이 낮으면 `high_risk`라고 경고한 것 중 FP가 많을 수 있습니다.


## 4. 여러 후보를 계속 학습하고 run 기록 만들기

이제 같은 과정을 후보별로 반복합니다. 각 후보는 새로 `fit()`되고, 같은 test split에서 평가됩니다. 반복 실행 결과는 tracking table에 쌓습니다.


In [ ]:
def train_evaluate_record(config: dict) -> dict:
    model = make_model(config)
    model.fit(train_df[FEATURE_COLUMNS], train_df["label"].eq(POSITIVE_LABEL).astype(int))

    scores = model.predict_proba(test_df[FEATURE_COLUMNS])[:, 1]
    predictions = [POSITIVE_LABEL if score >= float(config["threshold"]) else aiq.NEGATIVE_LABEL for score in scores]
    pred_binary = pd.Series(predictions).eq(POSITIVE_LABEL).astype(int)
    true_binary = test_df["label"].eq(POSITIVE_LABEL).astype(int)
    tn, fp, fn, tp = confusion_matrix(true_binary, pred_binary, labels=[0, 1]).ravel()

    return {
        "run_id": config["run_id"],
        "model_name": config["model_name"],
        "model_type": config["model_type"],
        "threshold": float(config["threshold"]),
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "accuracy": round(float(accuracy_score(true_binary, pred_binary)), 4),
        "precision": round(float(precision_score(true_binary, pred_binary, zero_division=0)), 4),
        "recall": round(float(recall_score(true_binary, pred_binary, zero_division=0)), 4),
        "f1": round(float(f1_score(true_binary, pred_binary, zero_division=0)), 4),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "artifact_uri": f"runs:/{config['run_id']}/model",
        "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }

run_table = pd.DataFrame([train_evaluate_record(row.to_dict()) for _, row in candidates.iterrows()])
run_table


**출력 확인**

이 표가 가장 중요한 출력입니다. 후보 모델을 여러 번 학습했고, 각 실행의 조건과 결과가 한 행씩 남았습니다.

MLflow UI에서는 이와 비슷하게 run 목록이 보입니다. 수강생은 여기서 “어떤 모델이 더 좋아 보이는가”보다 “비교에 필요한 조건이 같이 남았는가”를 먼저 확인해야 합니다.


In [ ]:
comparison_view = run_table.sort_values(
    by=["recall", "precision", "f1"],
    ascending=[False, False, False],
).loc[:, ["run_id", "model_name", "precision", "recall", "f1", "false_positive", "false_negative", "artifact_uri"]]
comparison_view


**출력 확인**

이 정렬은 예시일 뿐입니다. 의료 위험 탐지처럼 놓치면 위험한 문제에서는 recall을 우선할 수 있고, 불필요한 경고 비용이 큰 문제에서는 precision을 더 중요하게 볼 수 있습니다.

따라서 “최고 모델”은 metric 숫자 하나로 정하지 않고, 운영 목적과 FP/FN 비용을 함께 놓고 정합니다.


## 5. run 기록을 JSON artifact로 남기기

마지막으로 run table을 파일로 저장합니다. 이 파일은 작은 JSON tracker 역할을 합니다. MLflow가 없는 환경에서도 최소한 이런 기록은 남겨야 나중에 어떤 모델을 비교했는지 설명할 수 있습니다.


In [ ]:
tracking_record = {
    "tracking_type": "notebook_json_tracker",
    "dataset_source": source,
    "feature_columns": FEATURE_COLUMNS,
    "positive_label": POSITIVE_LABEL,
    "runs": run_table.to_dict(orient="records"),
}

output_path = Path("artifacts/experiments/chapter_02/training_tracking_runs.json")
try:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(tracking_record, indent=2, ensure_ascii=False), encoding="utf-8")
    saved_to = str(output_path)
except OSError:
    saved_to = "not_saved_in_this_runtime"

print("saved_to:", saved_to)
print("run_count:", len(tracking_record["runs"]))


**출력 확인**

`run_count`가 3이면 세 후보를 모두 학습하고 기록한 것입니다. `saved_to`가 파일 경로로 나오면 JSON artifact가 저장되었습니다.

이 JSON은 MLflow를 대체하는 최종 도구가 아닙니다. 다만 MLflow를 처음 보기 전에 “run마다 params, metrics, artifact URI를 남긴다”는 구조를 이해하기 위한 가장 작은 예제입니다.


In [ ]:
handoff = pd.DataFrame(
    [
        {"next_step": "03_precision_recall.ipynb", "what_to_check": "FP/FN, precision, recall을 손으로 해석합니다."},
        {"next_step": "04_read_metric_record.ipynb", "what_to_check": "저장된 metric record에 조건과 결과가 함께 남았는지 봅니다."},
        {"next_step": "local labs/ch02_model_quality/09_mlflow_tracking_lab.ipynb", "what_to_check": "로컬 Jupyter 환경에서 MLflow run 구조로 이어지는지 봅니다."},
        {"next_step": "demos/ch02_mlflow/01_run_with_docker_mlflow.sh", "what_to_check": "MLflow container를 띄우고 실제 tracking server에 기록합니다."},
    ]
)
handoff


**출력 확인**

이 notebook은 반복 학습과 기록의 감각을 만드는 단계입니다. 다음 notebook에서는 metric 계산을 더 천천히 보고, 그 다음에는 준비된 평가 기록과 MLflow 기록으로 이어집니다.
